In [ ]:
import mne
import pandas as pd

In [ ]:
hd = '/Volumes/evan_ext/EEG Analysis/'
mne.set_log_level('ERROR')

In [ ]:
def epoch_raw_data(file_path,max_freq=45, downsample_freq=250):
    """
    Load EEG, pick key channels + stim, filter, (optionally) downsample, and epoch around events.
    For .mff, the stim channel may be 'DIN4' or '40hz'—we detect it automatically.
    """
    file_path = str(file_path)  # in case a Path is passed
    # --- montage ---
    montage = mne.channels.read_custom_montage(hd + 'data/9_18AverageNet128_v1.sfp')

    # channels needed for analysis (EEG only; stim added dynamically)
    eeg_channels = ['E106', 'E7', 'E57', 'E101']

    # keep logs quiet
    mne.set_log_level('ERROR')

    if file_path.lower().endswith('.mff'):
        # Load
        raw = mne.io.read_raw_egi(
            file_path,
            eog=None, misc=None,
            preload=True,
            events_as_annotations=False
        )

        # --- determine stim channel (DIN4 preferred, else 40hz) ---
        chs = set(raw.ch_names)
        if 'DIN4' in chs:
            stim_ch = 'DIN4'
        elif '40hz' in chs:
            stim_ch = '40hz'
        else:
            raise RuntimeError(
                "No stim channel found. Expected 'DIN4' or '40hz' in the .mff file."
            )

        # --- pick available EEG + stim ---
        picks_to_keep = [ch for ch in eeg_channels if ch in chs] + [stim_ch]
        if stim_ch not in picks_to_keep:
            raise RuntimeError(f"Stim channel {stim_ch} not present after picking.")
        raw.pick(picks_to_keep, exclude=[])

        # --- filtering (EEG only) ---
        raw.filter(l_freq=1, h_freq=max_freq, picks='eeg')

        # --- set reference & montage ---
        # Only use refs that actually exist after picking
        ref_available = [ch for ch in ['E57', 'E101'] if ch in raw.ch_names]
        if len(ref_available) == 2:
            raw.set_eeg_reference(ref_channels=ref_available)  # type: ignore
        else:
            # fall back to average if either ref missing
            raw.set_eeg_reference('average')  # type: ignore

        raw.set_montage(montage)

        # --- (optional) downsample BEFORE event finding ---
        if downsample_freq:
            raw.resample(downsample_freq)

        # --- events & epochs ---
        events = mne.find_events(raw, stim_channel=stim_ch)
        epochs = mne.Epochs(
            raw,
            events,
            tmin=-0.5,
            tmax=1.0,
            baseline=(-0.1, 0),
            reject=dict(eeg=120e-6),
            detrend=1,
            preload=True,
        )

        # drop refs (if there) and the stim channel we used
        to_drop = [ch for ch in ['E57', 'E101', stim_ch] if ch in epochs.ch_names]
        if to_drop:
            epochs.drop_channels(to_drop)

        return epochs

    if file_path.lower().endswith('.set'):
        raw = mne.io.read_raw_eeglab(file_path, preload=True)
        # TODO: implement EEGLAB stim handling (often events are in raw.events/annotations)
        # For example:
        # events, event_id = mne.events_from_annotations(raw)
        # ... then epoch similarly ...
        raise NotImplementedError("EEGLAB (.set) handling not implemented yet.")

    raise ValueError("Unsupported file type. Expected a .mff or .set file.")

In [ ]:
from pathlib import Path

def exclude_flag(group, n, had_err=False):
    if had_err: return 1
    g = str(group).upper()
    if g in {"ASD", "PMS"}:  return 1 if (n or 0) < 25 else 0
    if g == "TD":            return 1 if (n or 0) < 40 else 0
    return 1 if (n or 0) <= 0 else 0

BASE = Path(f"/Volumes/evan_ext/EEG Analysis/analyses/Epoched Data/Final Epochs (max freq 45Hz)")

df = pd.read_csv("/Volumes/evan_ext/EEG Analysis/data/participant_info.csv")
df = df.rename(columns=lambda c: c.strip().strip("'").strip('"'))

for i, row in df.iterrows():
    pid   = str(row["Participant ID"]).strip()
    site  = str(row["Site"]).strip()
    group = str(row["Group"]).strip()
    mff   = str(row["MFF Path"]).strip()
    cur   = str(row.get("Epoch Files", "") or "").strip()


    out = BASE / site / group / f"{pid}_epoched_data.fif"
    out.parent.mkdir(parents=True, exist_ok=True)

    # --- check existing files ---
    resolved = None
    if cur and Path(cur).is_file():
        resolved = Path(cur)
        print(f"[SKIP] {pid}: existing Epoch Files path found ({cur})")
    elif out.is_file():
        resolved = out
        print(f"[SKIP] {pid}: canonical epoched file already exists ({out})")

    if resolved:
        df.at[i, "Epoch Files"] = str(resolved)
        continue

    # --- epoching ---
    print(f"[EPOCH] {pid}: starting epoching (Site={site}, Group={group})")
    n = None
    had_err = False
    try:
        mff_path = Path(mff)

        ep = epoch_raw_data(str(mff_path))
        n = int(len(ep))
        ep.save(str(out), overwrite=True)

        df.at[i, "Epoch Files"] = str(out)
        df.at[i, "Number of Samples"] = float(n)
        print(f"[DONE] {pid}: epoched successfully with {n} samples → {out}")
    except Exception as e:
        had_err = True
        print(f"[FAIL] {pid}: {type(e).__name__}: {e}")

    df.at[i, "Exclude"] = exclude_flag(group, n, had_err)

df.to_csv("/Volumes/evan_ext/EEG Analysis/data/participant_info.csv", index=False)
print("CSV updated.")

In [ ]:
df = pd.read_csv("/Volumes/evan_ext/EEG Analysis/data/participant_info.csv")
df = df.rename(columns=lambda c: c.strip().strip("'").strip('"'))
df = df[df['Site']== 'MSH']
df = df[df['Exclude']==0]
df = df[df['Number of Samples'] >= 40]

df = df[~df['Participant ID'].isin(['FF0261', 'FF0661'])] #hand remove subjects with problamatic waveforms due to high 40 hz power likely caused by audio leakage

study_xlsx = "/Volumes/evan_ext/EEG Analysis/data/StudyIDs_JS.xlsx"
study_df = pd.read_excel(study_xlsx,header=0)

df = df.merge(
    study_df[['FFID', 'Age_years','Sex','IQ_score','ADOS_comp_score']],     # only keep the needed columns
    how='left',                     # keep all rows from df, fill missing with NaN
    left_on='Participant ID',       # match key from df
    right_on='FFID'                 # match key from ff
)

df['IQ_score'] = pd.to_numeric(df['IQ_score'], errors='coerce')
df['ADOS_comp_score'] = pd.to_numeric(df['ADOS_comp_score'], errors='coerce')

# optionally drop the redundant 'FFID' column now that it's merged
df = df.drop(columns='FFID')
df = df[~df['Age_years'].isna()]
df

In [ ]:
import numpy as np
from neurodsp.spectral import compute_spectrum_welch

# =========================
# --- Helper Functions ----
# =========================
def get_ts(epoch: mne.Epochs):
    """Compute mean time series across channels and trials."""
    return np.mean(epoch.get_data(), axis=(0, 1))

def get_power_spectrum(ts):
    """Compute power spectrum from time series."""
    freqs, power = compute_spectrum_welch(ts,250)
    return freqs, power


def get_itpc_features(epoch: mne.Epochs, band_center=40, band_width=10):
    """Compute per-channel ITPC, average across channels manually."""
    fmin = band_center - band_width / 2.0
    fmax = band_center + band_width / 2.0
    freqs = np.linspace(fmin, fmax, 20)
    n_cycles = freqs / 2.0
    decim = 2

    itpc_list = []

    for ch_idx in range(len(epoch.ch_names)):
        _, itpc = mne.time_frequency.tfr_morlet(
            epoch, freqs=freqs, n_cycles=n_cycles, use_fft=True,
            return_itc=True, decim=decim, average=True, picks=[ch_idx]
        )
        itpc_list.append(itpc.data.squeeze())

    itpc_mean = np.mean(itpc_list, axis=0)
    
    return itpc_mean

def get_ersp_features(epoch: mne.Epochs, band_center=40, band_width=10):
    """Compute ERSP over all channels and trials"""
    fmin = band_center - band_width / 2.0
    fmax = band_center + band_width / 2.0
    freqs = np.linspace(fmin, fmax, 20)
    n_cycles = freqs / 2.0

    decim = 2
    baseline = (-.1, 0)
    baseline_mode = "logratio"

    power = mne.time_frequency.tfr_morlet(
        epoch, freqs=freqs, n_cycles=n_cycles, use_fft=True,
        return_itc=False, decim=decim, average=True
    )

    power.apply_baseline(baseline=baseline, mode=baseline_mode)
    power_features = power.data.squeeze().mean(axis=0)

    return power_features

def collate_ASSR_features(group:str):
    sub_df = df[df['Group'] == group]
    group_ts = []
    group_itpc = []
    group_ersp = []

    for epoch_path in sub_df['Epoch Files']:
        epoch = mne.read_epochs(epoch_path, preload=True)
        ts = get_ts(epoch)
        group_ts.append(ts)

        # Get ITPC for each subject
        itpc = get_itpc_features(epoch, band_center=40, band_width=10)
        group_itpc.append(itpc)

        ersp = get_ersp_features(epoch, band_center=40, band_width=10)
        group_ersp.append(ersp)
    
    return (group_ts, group_itpc, group_ersp)

In [ ]:
df['Epochs'] = df['Epoch Files'].map(lambda x: mne.read_epochs(x,preload=True))
df['TS features'] = df['Epochs'].map(lambda x: get_ts(x))
df['ITPC features'] = df['Epochs'].map(lambda x: get_itpc_features(x).mean(axis=0))
df['r_crit'] = df['Number of Samples'].map(lambda x: np.sqrt(-(1.0 /x) * np.log(0.5)))
df['ITPC_features_corrected'] = df['ITPC features'] - df['r_crit']
df['ERSP features'] = df['Epochs'].map(lambda x: get_ersp_features(x).mean(axis=0))
df

In [ ]:
from sklearn.linear_model import LinearRegression

def regress_out_age(feature_column, age_column):
    """
    Regress age from a dataframe column that contains feature vectors.
    Returns residuals with same shape as original feature vectors.
    """
    # Stack into matrix (n_subjects, n_features)
    X = np.vstack(feature_column.to_numpy())
    age = age_column.to_numpy()[:, None]  # shape (n_subjects, 1)

    model = LinearRegression()
    model.fit(age, X)

    X_pred = model.predict(age)
    residuals = X - X_pred

    return list(residuals)

In [ ]:
df['ITPC_features_age_resid'] = regress_out_age(
    df['ITPC features'],
    df['Age_years']
)

df['ITPC_features_corrected_age_resid'] = regress_out_age(
    df['ITPC_features_corrected'],
    df['Age_years']
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LinearRegression
from xgboost import XGBClassifier


def itpc_full_sensitivity(
        df,
        group1: str = "PMS",
        group2: str = "TD",
        n_bootstrap: int = 1000,
        random_state: int = 42,
    ):
    """
    Unified ITPC sensitivity analysis for group1 vs group2.

    Uses only ITPC-based features:
      - 'ITPC features'              (raw ITPC)
      - 'ITPC_features_corrected'    (trial-count–corrected ITPC)

    Outputs:
      - Baseline AUROCs (raw, corrected)
      - Age-residualized AUROCs (raw, corrected)
      - IQ-residualized AUROCs via bootstrap (raw, corrected)
      - Age+IQ–residualized AUROCs via bootstrap (raw, corrected)

    Steps:
      1) Filter df to group1 and group2.
      2) Remove non-TD participants with missing IQ_score.
      3) Build ITPC feature matrices (raw, corrected).
      4) Compute baseline AUROCs for raw and corrected.
      5) Residualize on age only (deterministic), compute AUROCs.
      6) For each bootstrap:
           - Impute missing IQ in TD ~ N(100, 15).
           - Residualize on IQ only (raw, corrected).
           - Residualize on age + IQ jointly (raw, corrected).
           - Compute LOO AUROCs.
      7) Return all AUROCs and bootstrap distributions.
    """

    # ---------------------------------------------------------
    # 1) Filter to group1 vs group2
    # ---------------------------------------------------------
    sub_df1 = df[df["Group"] == group1].copy()
    sub_df2 = df[df["Group"] == group2].copy()
    sub_df = pd.concat([sub_df1, sub_df2], axis=0).reset_index(drop=True)

    if "IQ_score" not in sub_df.columns:
        raise ValueError("Column 'IQ_score' not found in df.")

    # ---------------------------------------------------------
    # 2) Remove non-TD participants with missing IQ_score
    # ---------------------------------------------------------
    missing_iq = sub_df["IQ_score"].isna()
    td_mask = sub_df["Group"] == "TD"

    remove_mask = (~td_mask) & missing_iq
    removed_n = remove_mask.sum()
    if removed_n > 0:
        print(f"[ITPC Sensitivity] Removing {removed_n} non-TD participants with missing IQ_score.")
        sub_df = sub_df.loc[~remove_mask].reset_index(drop=True)

    # Recompute group splits and labels after filtering
    sub_df1 = sub_df[sub_df["Group"] == group1]
    sub_df2 = sub_df[sub_df["Group"] == group2]

    labels = np.hstack([
        np.ones(len(sub_df1)),
        np.zeros(len(sub_df2)),
    ])

    print(f"[ITPC Sensitivity] Participants after filtering: {len(labels)} "
          f"| {group1}={labels.sum()} | {group2}={(labels == 0).sum()}")

    # ---------------------------------------------------------
    # 3) Extract ITPC features (raw and corrected)
    # ---------------------------------------------------------
    if "ITPC features" not in sub_df.columns:
        raise ValueError("Column 'ITPC features' not found in df.")
    if "ITPC_features_corrected" not in sub_df.columns:
        raise ValueError("Column 'ITPC_features_corrected' not found in df.")

    X_raw = np.vstack(sub_df["ITPC features"].to_numpy())           # (n, p)
    X_corr = np.vstack(sub_df["ITPC_features_corrected"].to_numpy())# (n, p)

    # Age (always observed) and IQ (with NaNs in TD)
    age = sub_df["Age_years"].to_numpy()[:, None]  # shape (n, 1)
    iq = sub_df["IQ_score"].to_numpy()             # shape (n,)
    missing_mask = np.isnan(iq)

    print(f"[ITPC Sensitivity] Remaining missing IQ_score (TD only): {missing_mask.sum()}")

    # ---------------------------------------------------------
    # Helpers: classifier + LOO
    # ---------------------------------------------------------
    def make_clf(seed):
        return XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=seed, n_jobs=4, eval_metric="logloss"
        )

    def loo_auc(X, y, seed_offset=0):
        """
        Compute AUROC under LOO CV for feature matrix X and labels y.
        seed_offset can be used to vary random_state slightly if desired.
        """
        loo = LeaveOneOut()
        clf = make_clf(random_state + seed_offset)
        y_scores = cross_val_predict(
            clf, X, y, cv=loo, method="predict_proba", n_jobs=1
        )[:, 1]
        return roc_auc_score(y, y_scores)

    rng = np.random.default_rng(random_state)

    results = {}

    # ---------------------------------------------------------
    # 4) Baseline: raw and corrected ITPC (no residualization)
    # ---------------------------------------------------------
    auc_raw = loo_auc(X_raw, labels, seed_offset=0)
    auc_corr = loo_auc(X_corr, labels, seed_offset=1)

    print(f"[ITPC Sensitivity] Baseline ITPC_raw AUROC = {auc_raw:.3f}")
    print(f"[ITPC Sensitivity] Baseline ITPC_corrected AUROC = {auc_corr:.3f}")

    results["ITPC_raw"] = auc_raw
    results["ITPC_corrected"] = auc_corr

    # ---------------------------------------------------------
    # 5) Age-only residualization (deterministic)
    # ---------------------------------------------------------
    lr_age_raw = LinearRegression()
    lr_age_raw.fit(age, X_raw)
    X_raw_age_resid = X_raw - lr_age_raw.predict(age)

    lr_age_corr = LinearRegression()
    lr_age_corr.fit(age, X_corr)
    X_corr_age_resid = X_corr - lr_age_corr.predict(age)

    auc_raw_age = loo_auc(X_raw_age_resid, labels, seed_offset=2)
    auc_corr_age = loo_auc(X_corr_age_resid, labels, seed_offset=3)

    print(f"[ITPC Sensitivity] ITPC_raw (age-residualized) AUROC = {auc_raw_age:.3f}")
    print(f"[ITPC Sensitivity] ITPC_corrected (age-residualized) AUROC = {auc_corr_age:.3f}")

    results["ITPC_raw_age_resid"] = auc_raw_age
    results["ITPC_corrected_age_resid"] = auc_corr_age

    # ---------------------------------------------------------
    # 6) Bootstrapped IQ-only and Age+IQ residualization
    # ---------------------------------------------------------
    auc_raw_iq_list = []
    auc_corr_iq_list = []
    auc_raw_ageiq_list = []
    auc_corr_ageiq_list = []

    for b in range(n_bootstrap):
        # 6a) Impute missing IQ in TD from N(100, 15)
        iq_imp = iq.copy()
        if missing_mask.sum() > 0:
            iq_imp[missing_mask] = rng.normal(
                loc=100.0,
                scale=15.0,
                size=missing_mask.sum()
            )

        iq_imp_col = iq_imp[:, None]  # (n, 1)

        # --- IQ-only residualization ---
        lr_iq_raw = LinearRegression()
        lr_iq_raw.fit(iq_imp_col, X_raw)
        X_raw_iq_resid = X_raw - lr_iq_raw.predict(iq_imp_col)

        lr_iq_corr = LinearRegression()
        lr_iq_corr.fit(iq_imp_col, X_corr)
        X_corr_iq_resid = X_corr - lr_iq_corr.predict(iq_imp_col)

        auc_raw_iq = loo_auc(X_raw_iq_resid, labels, seed_offset=10 + b)
        auc_corr_iq = loo_auc(X_corr_iq_resid, labels, seed_offset=20 + b)

        auc_raw_iq_list.append(auc_raw_iq)
        auc_corr_iq_list.append(auc_corr_iq)


    auc_raw_iq_array = np.array(auc_raw_iq_list)
    auc_corr_iq_array = np.array(auc_corr_iq_list)

    def summarize_bootstrap(arr):
        mean = arr.mean()
        ci_lower = np.percentile(arr, 2.5)
        ci_upper = np.percentile(arr, 97.5)
        return mean, (ci_lower, ci_upper)

    raw_iq_mean, raw_iq_ci = summarize_bootstrap(auc_raw_iq_array)
    corr_iq_mean, corr_iq_ci = summarize_bootstrap(auc_corr_iq_array)

    print(f"[ITPC Sensitivity] ITPC_raw (IQ-residualized) AUROC = {raw_iq_mean:.3f} "
          f"[95% CI: {raw_iq_ci[0]:.3f}, {raw_iq_ci[1]:.3f}]")
    print(f"[ITPC Sensitivity] ITPC_corrected (IQ-residualized) AUROC = {corr_iq_mean:.3f} "
          f"[95% CI: {corr_iq_ci[0]:.3f}, {corr_iq_ci[1]:.3f}]")


    # Store everything in results
    results.update({
        "ITPC_raw_IQ_resid_mean": raw_iq_mean,
        "ITPC_raw_IQ_resid_ci95": raw_iq_ci,
        "ITPC_raw_IQ_resid_all": auc_raw_iq_array,

        "ITPC_corrected_IQ_resid_mean": corr_iq_mean,
        "ITPC_corrected_IQ_resid_ci95": corr_iq_ci,
        "ITPC_corrected_IQ_resid_all": auc_corr_iq_array,
    })

    return results

In [ ]:
res = itpc_full_sensitivity(df, group1="PMS", group2="TD", n_bootstrap=100, random_state=42)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import linregress
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


def age_matched_loocv(
        df,
        group1: str = "PMS",
        group2: str = "TD",
        n_per_group: int = 15,
        random_state: int = 42,
    ):
    """
    For each participant (LOO), train two classifiers:
      1. Global model  — all other participants (standard LOO)
      2. Age-matched model — n_per_group nearest by age from each group,
                             excluding the test participant

    Uses raw ITPC features only (no correction, no residualization).

    Returns a DataFrame with one row per participant containing:
      - group, age
      - global_prob      : P(group1) from global LOO model
      - matched_prob     : P(group1) from age-matched model
      - delta            : matched_prob - global_prob
      - mean_age_gap     : mean |age_train - age_test| across the 2*n_per_group
                           training participants in the age-matched model
    """

    sub_df = df[df["Group"].isin([group1, group2])].copy().reset_index(drop=True)

    X = np.vstack(sub_df["ITPC features"].to_numpy())
    ages = sub_df["Age_years"].to_numpy()
    groups = sub_df["Group"].to_numpy()
    labels = (groups == group1).astype(int)

    n = len(labels)
    g1_idx = np.where(groups == group1)[0]
    g2_idx = np.where(groups == group2)[0]

    print(f"[age_matched_loocv] n={n}  {group1}={len(g1_idx)}  {group2}={len(g2_idx)}")

    def make_clf(seed=random_state):
        return XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=seed, n_jobs=4, eval_metric="logloss",
        )

    global_probs  = np.full(n, np.nan)
    matched_probs = np.full(n, np.nan)
    mean_age_gaps = np.full(n, np.nan)

    for i in range(n):
        test_age = ages[i]

        # 1) Global LOO
        train_mask = np.ones(n, dtype=bool)
        train_mask[i] = False

        clf_global = make_clf(seed=random_state)
        clf_global.fit(X[train_mask], labels[train_mask])
        global_probs[i] = clf_global.predict_proba(X[[i]])[0, 1]

        # 2) Age-matched model
        g1_pool = g1_idx[g1_idx != i]
        g2_pool = g2_idx[g2_idx != i]

        g1_sorted = g1_pool[np.argsort(np.abs(ages[g1_pool] - test_age))][:n_per_group]
        g2_sorted = g2_pool[np.argsort(np.abs(ages[g2_pool] - test_age))][:n_per_group]

        train_idx = np.concatenate([g1_sorted, g2_sorted])

        if len(np.unique(labels[train_idx])) < 2:
            matched_probs[i] = np.nan
            mean_age_gaps[i] = np.nan
            continue

        clf_matched = make_clf(seed=random_state + i + 10000)
        clf_matched.fit(X[train_idx], labels[train_idx])
        matched_probs[i] = clf_matched.predict_proba(X[[i]])[0, 1]

        mean_age_gaps[i] = np.mean(np.abs(ages[train_idx] - test_age))

    results_df = pd.DataFrame({
        "participant_idx": np.arange(n),
        "group":           groups,
        "age":             ages,
        "label":           labels,
        "global_prob":     global_probs,
        "matched_prob":    matched_probs,
        "delta":           matched_probs - global_probs,
        "mean_age_gap":    mean_age_gaps,
    }).dropna()

    auc_global  = roc_auc_score(results_df["label"], results_df["global_prob"])
    auc_matched = roc_auc_score(results_df["label"], results_df["matched_prob"])
    r_probs     = np.corrcoef(results_df["global_prob"], results_df["matched_prob"])[0, 1]
    r_age_delta = np.corrcoef(results_df["mean_age_gap"], results_df["delta"])[0, 1]

    print(f"[age_matched_loocv] Global  AUROC = {auc_global:.3f}")
    print(f"[age_matched_loocv] Matched AUROC = {auc_matched:.3f}")
    print(f"[age_matched_loocv] Pearson r (global vs matched prob) = {r_probs:.3f}")
    print(f"[age_matched_loocv] Pearson r (mean age gap vs delta)  = {r_age_delta:.3f}")

    return results_df


def plot_age_matched_results(results_df, group1="PMS", group2="TD", save_path=None):
    """
    Two-panel figure:
      Left  — scatter of global_prob vs matched_prob, coloured by group
      Right — scatter of mean_age_gap vs delta (matched - global), coloured by group

    Adds linear regression p-values and R^2 for both panels.
    """
    colors = {group1: "#378ADD", group2: "#D85A30"}

    fig = plt.figure(figsize=(11, 4.8))
    gs  = gridspec.GridSpec(1, 2, wspace=0.35)

    clean = results_df[["group", "global_prob", "matched_prob", "mean_age_gap", "delta"]].dropna()

    # Linear model for panel A
    lm1 = linregress(clean["global_prob"], clean["matched_prob"])
    r2_1 = lm1.rvalue ** 2
    p1 = lm1.pvalue

    # Linear model for panel B
    lm2 = linregress(clean["mean_age_gap"], clean["delta"])
    r2_2 = lm2.rvalue ** 2
    p2 = lm2.pvalue

    print(f"[plot_age_matched_results] Panel A: matched ~ global")
    print(f"  slope = {lm1.slope:.4f}, intercept = {lm1.intercept:.4f}, R^2 = {r2_1:.3f}, p = {p1:.3e}")
    print(f"[plot_age_matched_results] Panel B: delta ~ mean_age_gap")
    print(f"  slope = {lm2.slope:.4f}, intercept = {lm2.intercept:.4f}, R^2 = {r2_2:.3f}, p = {p2:.3e}")

    # Panel A
    ax1 = fig.add_subplot(gs[0])
    for grp, sub in clean.groupby("group"):
        ax1.scatter(sub["global_prob"], sub["matched_prob"],
                    c=colors.get(grp, "gray"), label=grp,
                    s=55, alpha=0.8, linewidths=0.5, edgecolors="white", zorder=3)

    ax1.plot([0, 1], [0, 1], "--", color="gray", linewidth=0.8, alpha=0.5, zorder=2)
    ax1.set_xlim(-0.02, 1.02)
    ax1.set_ylim(-0.02, 1.02)
    ax1.set_xlabel("Global LOO predicted probability", fontsize=10)
    ax1.set_ylabel("Age-matched predicted probability", fontsize=10)
    ax1.set_title("A  Model consistency", fontsize=11, fontweight="normal", loc="left")
    ax1.legend(fontsize=9, framealpha=0.5)

    x1 = clean["global_prob"].to_numpy()
    x1_fit = np.linspace(x1.min(), x1.max(), 100)
    ax1.plot(x1_fit, lm1.slope * x1_fit + lm1.intercept, "-", color="gray", linewidth=1, alpha=0.7, zorder=2)

    ax1.text(0.04, 0.94, f"R² = {r2_1:.3f}\np = {p1:.2e}",
             transform=ax1.transAxes, fontsize=9, color="gray", va="top")

    # Panel B
    ax2 = fig.add_subplot(gs[1])
    for grp, sub in clean.groupby("group"):
        ax2.scatter(sub["mean_age_gap"], sub["delta"],
                    c=colors.get(grp, "gray"), label=grp,
                    s=55, alpha=0.8, linewidths=0.5, edgecolors="white", zorder=3)

    ax2.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5, zorder=2)

    x2 = clean["mean_age_gap"].to_numpy()
    x2_fit = np.linspace(x2.min(), x2.max(), 100)
    ax2.plot(x2_fit, lm2.slope * x2_fit + lm2.intercept, "-", color="gray", linewidth=1, alpha=0.7, zorder=2)

    ax2.set_xlabel("Mean age gap to training set (years)", fontsize=10)
    ax2.set_ylabel("Δ probability  (matched − global)", fontsize=10)
    ax2.set_title("B  Age-gap sensitivity", fontsize=11, fontweight="normal", loc="left")
    ax2.legend(fontsize=9, framealpha=0.5)

    ax2.text(0.04, 0.94, f"R² = {r2_2:.3f}\np = {p2:.2e}",
             transform=ax2.transAxes, fontsize=9, color="gray", va="top")

    fig.suptitle("Age-matched vs global LOO — raw ITPC", fontsize=12,
                 fontweight="normal", y=1.01)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig

In [ ]:
results_df = age_matched_loocv(df, group1="PMS", group2="TD", n_per_group=24)
fig = plot_age_matched_results(results_df, group1="PMS", group2="TD",
                               save_path="age_matched_consistency.svg")

In [ ]:
def get_itpc_flattened(epoch: mne.Epochs, band_center=40, band_width=10):
    """Compute per-channel ITPC, average across channels manually."""
    fmin = band_center - band_width / 2.0
    fmax = band_center + band_width / 2.0
    freqs = np.linspace(fmin, fmax, 20)
    n_cycles = freqs / 2.0
    decim = 2

    itpc_list = []

    for ch_idx in range(len(epoch.ch_names)):
        _, itpc = mne.time_frequency.tfr_morlet(
            epoch, freqs=freqs, n_cycles=n_cycles, use_fft=True,
            return_itc=True, decim=decim, average=True, picks=[ch_idx]
        )
        itpc_list.append(itpc.data.squeeze())

    itpc_mean = np.mean(itpc_list, axis=0)
    
    return itpc_mean.flatten()

df['Flat ITPC'] = df['Epochs'].map(lambda x: get_itpc_flattened(x))
df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix

# ================================================================
# SETTINGS
# ================================================================
groups = ["TD", "ASD", "PMS"]
group_to_color = {"TD": "tab:blue", "ASD": "tab:green", "PMS": "tab:red"}
cluster_colors = {0: "grey", 1: "maroon"}

# ================================================================
# FEATURE MATRICES (FULL COHORT)
# ================================================================
X_ITPC = np.vstack(df["Flat ITPC"].to_numpy())

age = df["Age_years"].to_numpy()[:, None]
sex = df["Sex"].map({"M":0,"F":1}).to_numpy()[:, None]
cov = np.hstack([age, sex])

X_FULL = np.hstack([X_ITPC, cov])

true_labels = df["Group"].to_numpy()

# ================================================================
# FUNCTION: RUN UMAP + K-MEANS ON FULL COHORT
# ================================================================
def full_umap_and_kmeans(features):
    um = umap.UMAP(random_state=42)

    emb = um.fit_transform(features)

    km = KMeans(n_clusters=2, random_state=42, n_init="auto")
    clusters = km.fit_predict(emb)

    return emb, clusters

# Run models:
emb_ITPC, cl_ITPC   = full_umap_and_kmeans(X_ITPC)
emb_FULL, cl_FULL   = full_umap_and_kmeans(X_FULL)

# ================================================================
# PLOTTING HELPERS
# ================================================================
def plot_clusters(ax, emb, clusters, title):
    for c in [0, 1]:
        idx = np.where(clusters == c)[0]
        ax.scatter(
            emb[idx, 0], emb[idx, 1],
            c=cluster_colors[c], s=70,
            edgecolor="k", label=f"Cluster {c}"
        )
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend()

def plot_true_labels(ax, emb, labels, title):
    for g in groups:
        idx = np.where(labels == g)[0]
        ax.scatter(
            emb[idx, 0], emb[idx, 1],
            c=group_to_color[g], s=70,
            edgecolor="k", label=g
        )

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend()

# ================================================================
# BUILD 2×2 FIGURE
# ================================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Top row = ITPC only
plot_clusters(
    axes[0,0], emb_ITPC, cl_ITPC,
    "ITPC Only — K-means Clusters"
)

plot_true_labels(
    axes[0,1], emb_ITPC, true_labels,
    "ITPC Only — True Group Labels"
)

# Bottom row = ITPC + Age + Sex
plot_clusters(
    axes[1,0], emb_FULL, cl_FULL,
    "ITPC + Age + Sex — K-means Clusters"
)

plot_true_labels(
    axes[1,1], emb_FULL, true_labels,
    "ITPC + Age + Sex — True Group Labels"
)

plt.tight_layout()
plt.show()

# ================================================================
# PMS CLASSIFICATION PERFORMANCE (FULL COHORT)
# ================================================================
def compute_pms_performance(cluster_labels, true_labels):
    mask = np.isin(true_labels, ["TD", "PMS"])
    binary_true = (true_labels[mask] == "PMS").astype(int)
    cl = cluster_labels[mask]

    results = {}
    for cluster_id in [0, 1]:
        pred = (cl == cluster_id).astype(int)

        tn, fp, fn, tp = confusion_matrix(binary_true, pred).ravel()

        sens = tp/(tp+fn) if (tp+fn)>0 else np.nan
        spec = tn/(tn+fp) if (tn+fp)>0 else np.nan

        results[f"Cluster {cluster_id}"] = (sens, spec)

    return results

# ITPC only
perf_itpc = compute_pms_performance(cl_ITPC, true_labels)

# ITPC + Age + Sex
perf_full = compute_pms_performance(cl_FULL, true_labels)

print("\n=======================================================")
print(" PMS Classification Based on Clusters")
print("=======================================================\n")

print("ITPC ONLY model:")
for cl, (sens, spec) in perf_itpc.items():
    print(f"  {cl}: Sensitivity={sens:.3f}, Specificity={spec:.3f}")
print()

print("ITPC + Age + Sex model:")
for cl, (sens, spec) in perf_full.items():
    print(f"  {cl}: Sensitivity={sens:.3f}, Specificity={spec:.3f}")
print()


In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, confusion_matrix

# ------------------------------------------------
# Fixed 2-cluster GMM
# ------------------------------------------------
def run_fixed_two_cluster(features, random_state=42):
    gm = GaussianMixture(n_components=2, random_state=random_state)
    return gm.fit_predict(features)

# ------------------------------------------------
# Bootstrap clustering stability + PMS vs TD metrics
# ------------------------------------------------
def bootstrap_fixed_clustering(
        X, true_labels,
        n_boot=200, sample_frac=0.8,
        random_state=42
    ):
    """
    X:           (n_samples, n_features) feature matrix
    true_labels: array-like of group labels (e.g., "TD", "ASD", "PMS")

    Returns:
        ari_scores:  array of ARI values vs reference clustering
        sens_list:   array of best sensitivity for PMS vs TD
        spec_list:   array of best specificity for PMS vs TD
    """
    rng = np.random.default_rng(random_state)
    n = X.shape[0]

    # Reference clustering on full data
    base_labels = run_fixed_two_cluster(X, random_state=random_state)

    ari_scores = []
    sens_list = []
    spec_list = []

    for b in range(n_boot):
        # Subsample indices
        sub_size = max(2, int(sample_frac * n))
        idx = rng.choice(n, size=sub_size, replace=False)

        X_sub = X[idx]
        tl_sub = true_labels[idx]

        # New clustering on the subsample
        sub_labels = run_fixed_two_cluster(
            X_sub, random_state=random_state + b
        )

        # --- ARI vs reference clustering on same indices ---
        ari = adjusted_rand_score(base_labels[idx], sub_labels)
        ari_scores.append(ari)

        # --- PMS vs TD performance in this subsample ---
        mask = np.isin(tl_sub, ["TD", "PMS"])
        if mask.sum() == 0:
            continue

        tl_pms = tl_sub[mask]
        # Need both PMS and TD present
        if len(np.unique(tl_pms)) < 2:
            continue

        binary_true = (tl_pms == "PMS").astype(int)
        cl = sub_labels[mask]

        best_j = -np.inf
        best_sens = np.nan
        best_spec = np.nan

        # Evaluate both cluster assignments as potential "PMS cluster"
        for cid in [0, 1]:
            pred = (cl == cid).astype(int)
            tn, fp, fn, tp = confusion_matrix(binary_true, pred).ravel()

            sens = tp/(tp+fn) if (tp+fn) > 0 else np.nan
            spec = tn/(tn+fp) if (tn+fp) > 0 else np.nan

            if np.isnan(sens) or np.isnan(spec):
                continue

            j = sens + spec - 1  # Youden's J
            if j > best_j:
                best_j = j
                best_sens = sens
                best_spec = spec

        if best_j > -np.inf:
            sens_list.append(best_sens)
            spec_list.append(best_spec)

    return (
        np.array(ari_scores),
        np.array(sens_list),
        np.array(spec_list),
    )

# ================================================================
# RUN BOOTSTRAP ANALYSES
# ================================================================
n_boot = 200
sample_frac = 0.8

ari_itpc, sens_itpc, spec_itpc = bootstrap_fixed_clustering(
    X_ITPC, true_labels,
    n_boot=n_boot, sample_frac=sample_frac,
    random_state=42
)

ari_full, sens_full, spec_full = bootstrap_fixed_clustering(
    X_FULL, true_labels,
    n_boot=n_boot, sample_frac=sample_frac,
    random_state=123
)

def summarize_bootstrap(name, ari, sens, spec):
    print(f"\n{name}")
    if len(ari) > 0:
        print(f"  ARI: mean={ari.mean():.3f}, 95% CI=({np.percentile(ari, 2.5):.3f}, {np.percentile(ari, 97.5):.3f})")
    else:
        print("  ARI: no valid bootstrap samples.")

    if len(sens) > 0:
        print(f"  Sensitivity: mean={sens.mean():.3f}, 95% CI=({np.percentile(sens, 2.5):.3f}, {np.percentile(sens, 97.5):.3f})")
    else:
        print("  Sensitivity: no valid bootstrap samples.")

    if len(spec) > 0:
        print(f"  Specificity: mean={spec.mean():.3f}, 95% CI=({np.percentile(spec, 2.5):.3f}, {np.percentile(spec, 97.5):.3f})")
    else:
        print("  Specificity: no valid bootstrap samples.")

print("\n=======================================================")
print(" GM Cluster Stability + PMS Classification (Bootstrap)")
print("=======================================================")

summarize_bootstrap("ITPC ONLY", ari_itpc, sens_itpc, spec_itpc)
summarize_bootstrap("ITPC + Age + Sex", ari_full, sens_full, spec_full)
print()